# ContractRisk — calibración de juez por API

Runtime estándar de Colab (CPU suficiente; no necesita T4). Candidato fijado gemini-3.8-flash, sin selección automática de otro modelo ni modo de pago. Verifica acceso al modelo antes de generar. Los datos enviados son únicamente las veinte anclas sintéticas previamente aprobadas, no el gold final.

Requiere secretos GEMINI_API_KEY y WANDB_API_KEY con acceso habilitado para ESTE notebook. No pegar claves en celdas o salidas. En Google AI Studio, comprobar que el proyecto de esa clave indica Free Tier y no tiene facturación activada. Si está pagado o no puede comprobarse, detenerse y consultar antes de ejecutar.

La celda de confirmación pedirá escribir GRATUITO. Esto es una declaración del titular, NO una detección automática de facturación. El registro de reservas de gasto y la marca de inicio se guardan en una carpeta de Drive compartida entre sesiones; no contienen claves ni preguntas. No borrar la marca para repetir una corrida: primero auditar cualquier resultado parcial. Presupuesto total autorizado USD 5; esta calibración se limita además a USD 1 de estimación conservadora, no de factura verificada.

Primero hasta nueve juicios piloto, empezando por la atribución falsa. Si fallan, se detiene. Si pasan, se reutilizan esas notas para completar 38 juicios. Se conserva nota original; un JSON válido no prueba buen juicio. No ejecuta confirmación nueva ni evaluación final ni modifica main/Space. No se guarda razonamiento privado.


In [ ]:
import os, sys, json, subprocess, tempfile
from pathlib import Path
from google.colab import userdata

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_api_judge_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
# Required by historical-integrity unit tests; no inference from that commit is rerun.
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin',
    '6ec8dcdf22433c632b5eb0e5539401c33b9c923d'], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)
for name in ['GEMINI_API_KEY', 'WANDB_API_KEY']:
    try:
        value = userdata.get(name)
        if not value:
            raise ValueError('Empty secret')
        os.environ[name] = value
        del value
    except Exception:
        raise RuntimeError('Habilita el acceso de ESTE notebook al secreto ' + name + '. No pegues la clave.') from None

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError('Proceso detenido; conservar resultados y consultar ' + logfile)


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'final/requirements-api-judge.txt'], 'installation.log')
run_visible([sys.executable, '-c', "from pathlib import Path; from final.run_judge_calibration import verify_review; verify_review(Path('final/eval/calibration_review_v1')); print('Anclas aprobadas y hashes comprobados.')"], 'review_check.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUDGET_DIR = Path('/content/drive/MyDrive/ContractRisk_final_budget')
confirmation = input('Confirma que el proyecto de GEMINI_API_KEY es Free Tier y NO tiene facturación activada. Escribe GRATUITO para continuar: ')
assert confirmation.strip() == 'GRATUITO', 'No se hará ninguna petición API sin confirmar el proyecto gratuito.'
del confirmation
print('Registro persistente de presupuesto en Drive. No borrar para repetir corridas.')


In [ ]:
RESULTS = ROOT / 'gemini_calibration_results'
run_visible([sys.executable, '-u', 'final/run_gemini_calibration.py', '--output', str(RESULTS),
    '--budget-dir', str(BUDGET_DIR), '--confirm-free-project', '--require-wandb'], 'gemini_calibration.log')


## Interpretación y conservación

completed_pilot_failed es fallo de puntuación del candidato, no necesariamente fallo técnico. stopped_api_or_format_failure puede indicar cuota, red, formato o presupuesto: no implica mal juicio en casos no ejecutados. failed puede ser falta de acceso al modelo o comprobación previa. No cambiar de modelo, activar facturación ni repetir automáticamente si ocurre 403/404/429.

completed_regression_passed_confirmation_pending tampoco valida todavía al juez: faltan auditoría humana de motivos y confirmación nueva revisada. Si el modelo no está disponible en tu cuenta, descarga igualmente los resultados para decidir cómo proceder. El coste registrado es una estimación conservadora con tarifas de referencia; no una factura ni prueba del estado de facturación. La marca de Drive evita repetir por accidente; no la borres.


In [ ]:
RESULTS = ROOT / 'gemini_calibration_results'
for name in ['metadata.json', 'pilot_summary.json', 'regression_summary.json', 'partial_regression_summary.json']:
    path = RESULTS / name
    if path.exists():
        print(name, path.read_text(encoding='utf-8'))


In [ ]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for logfile in ROOT.glob('*.log'):
    shutil.copy2(logfile, RESULTS / logfile.name)
for name in ['gemini_calibration_config.json', 'api_budget_policy.json']:
    shutil.copy2(ROOT / 'final' / name, RESULTS / name)
reviewed = RESULTS / 'reviewed_calibration_pack'
if not reviewed.exists():
    shutil.copytree(ROOT / 'final/eval/calibration_review_v1', reviewed)
package = shutil.make_archive(str(ROOT / 'contractrisk_final_juez_api_resultados'), 'zip', RESULTS)
files.download(package)
